# 5. Model track C - Random Forest & XGBoost

## 5.1. Thiết lập môi trường

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent  # notebooks/ -> gốc dự án
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

FIG_DIR = project_root / "figures" / "model_track_c"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix

from src import data, metrics

## 5.2. Nạp dữ liệu đã xử lý

In [3]:
X, y, _ = data.load_processed()
fold = data.load_folds()

## 5.3. Khai báo mô hình

In [4]:
models = {
    "Random Forest": RandomForestClassifier(random_state=data.RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(random_state=data.RANDOM_STATE, eval_metric="mlogloss", n_jobs=-1),
}

## 5.4. Huấn luyện & đánh giá bằng Cross-Validation

Gom thêm dự đoán out-of-fold (mỗi dòng dữ liệu được dự đoán đúng 1 lần, khi nó nằm ở tập validation) để dùng vẽ confusion matrix và trích xuất độ quan trọng của đặc trưng (feature importance) ở phần sau — không rò rỉ dữ liệu vì mỗi fold luôn dự đoán trên phần nó **chưa** thấy khi train. Bắt buộc dùng `clone(base_model)` để tạo bản sao **chưa huấn luyện** ở mỗi fold.

In [5]:
rows = []
oof_true = {name: [] for name in models}
oof_pred = {name: [] for name in models}
feature_importances = {name: [] for name in models}

for f, tr, va in data.iter_folds(fold):
    for name, base_model in models.items():
        model = clone(base_model)
        model.fit(X.iloc[tr], y.iloc[tr])
        proba = model.predict_proba(X.iloc[va])

        rows.append({"model": name, "fold": f, **metrics.compute_metrics(y.iloc[va], proba)})
        oof_true[name].append(y.iloc[va])
        oof_pred[name].append(proba.argmax(axis=1))
        
        if hasattr(model, "feature_importances_"):
            feature_importances[name].append(model.feature_importances_)

scores = pd.DataFrame(rows)
scores.head()

,model,fold,log_loss,accuracy,macro_f1
0,Random Forest,0,0.506994,0.847396,0.575549
1,XGBoost,0,0.448437,0.842708,0.639104
2,Random Forest,1,0.566687,0.853646,0.583155
3,XGBoost,1,0.421962,0.855729,0.623734
4,Random Forest,2,0.509077,0.844271,0.572629


## 5.5. Lưu kết quả CV

In [6]:
scores.to_csv("../results/scores_track_c.csv", index=False)
scores

,model,fold,log_loss,accuracy,macro_f1
0,Random Forest,0,0.506994,0.847396,0.575549
1,XGBoost,0,0.448437,0.842708,0.639104
2,Random Forest,1,0.566687,0.853646,0.583155
3,XGBoost,1,0.421962,0.855729,0.623734
4,Random Forest,2,0.509077,0.844271,0.572629
5,XGBoost,2,0.427903,0.849479,0.640737
6,Random Forest,3,0.470299,0.843750,0.570694
7,XGBoost,3,0.424350,0.846354,0.633817
8,Random Forest,4,0.553487,0.836979,0.555325
9,XGBoost,4,0.437685,0.845313,0.646938


## 5.6. Bảng tổng hợp kết quả (mean ± std)

In [7]:
summary = metrics.summarize_scores(scores)
summary

log_loss            accuracy            macro_f1          
                   mean       std      mean       std      mean       std
model                                                                    
Random Forest  0.521309  0.038895  0.845208  0.006056  0.571470  0.010195
XGBoost        0.432067  0.010938  0.847917  0.004996  0.636866  0.008706

## 5.7. Confusion Matrix (Out-of-Fold)

Ma trận nhầm lẫn được tính trên dự đoán out-of-fold (gộp cả 5 fold), chuẩn hóa theo hàng (`normalize="true"`) để xem tỉ lệ dự đoán đúng/nhầm của từng lớp thực tế.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, name in zip(axes, models):
    y_true_all = pd.concat(oof_true[name])
    y_pred_all = np.concatenate(oof_pred[name])
    cm = confusion_matrix(y_true_all, y_pred_all, labels=data.LABELS, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=data.CLASS_ORDER, yticklabels=data.CLASS_ORDER, ax=ax, cbar=False)
    ax.set_title(name)
    ax.set_xlabel("Dự đoán")
    ax.set_ylabel("Thực tế")
plt.tight_layout()
fig.savefig(FIG_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.8. Phân tích độ quan trọng đặc trưng (Feature Importance)

Trực quan hóa Top 15 đặc trưng đóng góp lớn nhất vào mô hình Random Forest và XGBoost trung bình qua 5 fold.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
feature_names = X.columns

for ax, name in zip(axes, models):
    mean_imp = np.mean(feature_importances[name], axis=0)
    imp_df = pd.DataFrame({"feature": feature_names, "importance": mean_imp}).sort_values("importance", ascending=False).head(15)
    sns.barplot(data=imp_df, x="importance", y="feature", ax=ax, hue="feature", legend=False, palette="viridis")
    ax.set_title(f"Top 15 đặc trưng quan trọng - {name}")
    ax.set_xlabel("Importance Score")
    ax.set_ylabel("Đặc trưng")

plt.tight_layout()
fig.savefig(FIG_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.9. So sánh trực quan giữa 2 mô hình

Biểu đồ cột so sánh trung bình 3 chỉ số (log loss, accuracy, macro-F1) giữa Random Forest và XGBoost qua 5 fold.

In [10]:
plot_df = summary.xs("mean", axis=1, level=1).reset_index()
plot_df = plot_df.melt(id_vars="model", var_name="metric", value_name="value")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=plot_df, x="metric", y="value", hue="model", ax=ax)
ax.set_title("So sánh Random Forest vs XGBoost (trung bình 5-fold)")
plt.tight_layout()
fig.savefig(FIG_DIR / "metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.10. Nhận xét

Qua 5-fold CV, mô hình **XGBoost** vượt trội hơn hẳn **Random Forest** ở cả 3 chỉ số đánh giá:
- **Log Loss:** XGBoost đạt **0.432 ± 0.011** (so với **0.521 ± 0.039** của Random Forest). Mức log loss của XGBoost giảm đáng kể nhờ thuật toán Gradient Boosting tối ưu hóa trực tiếp hàm mất mát log loss đa lớp (eval_metric='mlogloss') và điều chỉnh xác suất đầu ra mịn hơn.
- **Accuracy:** XGBoost đạt **0.848 ± 0.005** (so với **0.845 ± 0.006** của Random Forest).
- **Macro F1:** XGBoost đạt **0.637 ± 0.009** (cao hơn hẳn **0.571 ± 0.010** của Random Forest). Điều này cho thấy XGBoost phân loại lớp hiếm `CL` tốt hơn nhiều nhờ khả năng tập trung vào các mẫu khó phân loại ở từng cây boosting tiếp theo.

**Đặc trưng quan trọng (Feature Importance):**
Cả hai mô hình đều xác định các chỉ số xét nghiệm gan chính như `Bilirubin_Level_log`, `Prothrombin_Time_log`, `Copper_Level_log`, `Albumin_Level` và `AST_Level_log` đóng vai trò quan trọng nhất trong việc dự đoán kết cục của bệnh nhân.

**Kết luận:** Trong họ mô hình Track C (Tree-based Ensembles), **XGBoost** là mô hình tốt nhất tính tới thời điểm hiện tại đối với chỉ số log loss của bài toán.